Das nachfolgende ist nur im Playground mit "falschem" dev notwendig.

In [1]:
from typing import TypedDict 

class EducationData(TypedDict): 
    degree: str 
    field_of_study: str 

class CVData(TypedDict): 
    #personal_info: PersonalData 
    #industry_skills: list[str] 
    #non_industry_skills: list[str] 
    education: list[EducationData] 
    #professional_experience: list[ProfessionalExperienceData] 

class RequirementsData(TypedDict): 
    #job_title: str 
    #industry_skills: list[str] 
    #non_industry_skills: list[str] 
    education: list[EducationData]
    #professional_experience: ProfessionalExperienceData

In [7]:
import json
import sys
import os
import pandas as pd
from typing import cast

# sys.path.append(os.path.abspath("../"))
# from matching.match_requirements import Model
# from src.config import CV_OUTPUT_DIR_MATCHING

In [9]:
ad_path = os.path.abspath("../LLM-parsed_CVs/requirements")
ad_path

file_path = os.path.join(ad_path, "B-Stellenbeschreibung_0.json")

with open(file_path, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

raw_education = raw_data["education"]

education = [
    {
        "degree": item["field"],
        "field_of_study": item["institution"],
    }
    for item in raw_education
]

ad_data = cast(list[EducationData], education)
ad_data

[{'degree': '', 'field_of_study': 'kaufmännische Ausbildung'},
 {'degree': '', 'field_of_study': 'technische Ausbildung'},
 {'degree': '', 'field_of_study': 'Lebensmitteltechnologie'}]

In [10]:
ad_list = ["kaufmännische",
           "technische",
           "Lebensmitteltechnologie"]

In [18]:
# model = Model("google-bert/bert-base-german-cased")

from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity
import torch
import torch.nn.functional as F
from sentence_transformers.util import cos_sim


tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-german-cased")
model = AutoModel.from_pretrained("google-bert/bert-base-german-cased")

In [ ]:
def extract_education(applicant_data: str) -> EducationData:
    """ Extract education from applicant file

    :param applicant_data: string that contains the file path to the applicant's json-file
    :type applicant_data: str
    :return: EducationData element with education items of an applicant
    :rtype: EducationData
    """

    # read data from json
    with open(applicant_data, "r", encoding="utf-8") as f:
        cv_dict = json.load(f)
    
    # extract education into EducationData
    raw_education = cv_dict["education"]
    education = [
        {
            "degree": item["graduated"],
            "field_of_study": item["field_of_study"],
        }
        for item in raw_education
    ]

    cv_data = cast(list[EducationData], education)
    return cv_data

def extract_edu_requirements(requirement_data: str) -> RequirementsData:
    """ Extract education from requirements file

    :param requirement_data: string that contains the file path to the requirements json-file
    :type requirement_data: str
    :return: RequirementsData element with education items of the requirement
    :rtype: RequirementsData    
    """
    
    # read data from json
    with open(requirement_data, "r", encoding="utf-8") as f:
        req_dict = json.load(f)
    
    # extract education into RequirementsData
    raw_req = req_dict["education"]
    edu_req = [
        {
            "degree": item["field"],
            "field_of_study": item["institution"],
        }
        for item in raw_req
    ]

    req_data = cast(list[RequirementsData], edu_req)
    return req_data


def embed_batch(texts: list[EducationData]):
    inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True)

    with torch.no_grad():
        outputs = model(**inputs)
    
    embeddings = outputs.last_hidden_state[:, 0, :]
    return embeddings


def compare_education(education_list: list[EducationData], education_requirements: list[EducationData]) -> dict:
    """ Compare education from an applicant with requirements

    :param education_list: contains education items of an applicant
    :type education_list: list[EducationData]
    :param education_requirements: contains education requirements for a position
    :type education_requirements: list[EducationData]
    :returns: matrix of matching scores 
    :rtype: dict
    """

    # get embeddings
    req_emb = embed_batch(education_requirements)
    cv_emb = embed_batch(education_list)

    # pairwise similarity
    # score_matrix = {}

    # for i, cv_item in enumerate(education_list):
    #     score_matrix[cv_item] = {}
    #     for j, req_item in enumerate(education_requirements):
    #         # similarity score
    #         req_vec = req_emb[j].reshape(1, -1)
    #         cv_vec = cv_emb[i].reshape(1, -1)
    #         score = float(cosine_similarity(req_vec, cv_vec).item())
    #         score_matrix[cv_item][req_item] = score
    
    score_matrix = cos_sim(cv_emb, req_emb)

    return score_matrix


def get_score(score_matrix: dict) -> float:
    """ Calculate final score 

    :param score_matrix: dictionary that contains all scores from the CV education and all requirements
    :type score_matrix: dict
    :return: final score = sum of all maximum of each education item
    :rtype: float
    """
    print(score_matrix.shape)

    
    max_similarities = torch.max(score_matrix, dim=0).values
    average_similarity = torch.mean(max_similarities).item()
    
    return average_similarity


def calculate_score(applicant_data, requirement_data):
    """ 
    
    """
    # Extract education strings
    # education_list = extract_education(applicant_data)

    # Extract requirement strings
    # education_requirements = extract_edu_requirements(requirement_data)

    # Build pairwise comparison matrix
    score_matrix = compare_education(applicant_data, requirement_data)



    # Compute final score
    education_score = get_score(score_matrix)

    return {education_score}


In [31]:
cv_edu = ["Lebensmitteltechnologie",
          "Grundkurs für Labor",
          "Lehrgang General Management Region Pinzgau",
          "Technische Seminare",
          "QM- und Management-Schulungen"
    ]

calculate_score(cv_edu, ad_list)

torch.Size([5, 3])


{0.8715483546257019}

In [ ]:
from sentence_transformers import SentenceTransformer, util

# Load pretrained German BERT
model = SentenceTransformer("bert-base-german-cased")

# Your texts
text1 = "Das ist ein Beispiel."
text2 = "Dies ist ein Musterbeispiel."
text3 = "Das Wetter ist heute schön."

# Embed all texts
emb1 = model.encode(text1, convert_to_tensor=True)
emb2 = model.encode(text2, convert_to_tensor=True)
emb3 = model.encode(text3, convert_to_tensor=True)

# Compute similarity scores
sim12 = util.cos_sim(emb1, emb2)
sim13 = util.cos_sim(emb1, emb3)

print("Similarity text1 vs text2:", float(sim12))
print("Similarity text1 vs text3:", float(sim13))


In [ ]:
def calculate_score(education_list: list[EducationData], education_requirements: list[EducationData]):
    """ 
    
    """
    # embeddings
    education_emb = 
    req_emb = 

    # Build pairwise comparison matrix
    score_matrix = compare_education(education_list, education_requirements)

    # Print comparison matrix
    print(pd.DataFrame(score_matrix).T)

    # Compute final score
    education_score = get_score(score_matrix)

    return {education_score}